# 07 · Route resilience

Este notebook responde **Q4**: quais rotas são mais resilientes ou mais vulneráveis quando a chuva aparece,
combinando variação de demanda, piora de headway, perda de velocidade e gap entre serviço observado e programado.

**Janela usada aqui:** a janela integrada de **19 dias**.


In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', '{:,.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.figsize': (12, 5), 'figure.dpi': 120})


def locate_project_root() -> Path:
    start = Path.cwd().resolve()
    for base in [start, *start.parents]:
        if (base / 'data' / 'derived').exists():
            return base
    raise FileNotFoundError('Could not locate the repo root from the current working directory.')


PROJECT_ROOT = locate_project_root()
DERIVED = PROJECT_ROOT / 'data' / 'derived'
FIGURES = DERIVED / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
WEATHER_ORDER = ['Clear', 'Light Rain', 'Moderate Rain', 'Heavy Rain / Storm']


def savefig(name: str) -> Path:
    path = FIGURES / name
    plt.tight_layout()
    plt.savefig(path, bbox_inches='tight')
    print(f'Saved figure: {path}')
    return path


In [ ]:
from scipy.stats import spearmanr
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

integrated = pd.read_parquet(DERIVED / 'integrated_route_hour.parquet')
base = integrated.dropna(subset=['observed_trip_count']).copy()
base = base.loc[base['coverage_flag'].fillna('ok') != 'partial_day'].copy()
base['rain_flag'] = np.where(base['rain_mm'].fillna(0) > 0, 'Rain', 'Clear')
print('=== Sample ===')
print('rows:', len(base))
print('unique routes:', base['route_norm'].nunique())
print('unique days:', pd.to_datetime(base['date']).dt.date.nunique())


In [ ]:
route_weather = (
    base.groupby(['route_norm', 'operator', 'rain_flag'], observed=True)
    .agg(
        boardings=('boardings', 'mean'),
        headway_p50=('headway_p50', 'mean'),
        speed_p50=('speed_p50', 'mean'),
        service_gap_index=('service_gap_index', 'mean'),
        observed_vs_scheduled_trip_ratio=('observed_vs_scheduled_trip_ratio', 'mean'),
        route_crosswalk_confidence=('route_crosswalk_confidence', 'first'),
    )
    .reset_index()
)

clear = route_weather.loc[route_weather['rain_flag'] == 'Clear'].copy()
rain = route_weather.loc[route_weather['rain_flag'] == 'Rain'].copy()
resilience = clear.merge(rain, on=['route_norm', 'operator', 'route_crosswalk_confidence'], suffixes=('_clear', '_rain'))
resilience['demand_delta_pct'] = 100 * (resilience['boardings_rain'] / resilience['boardings_clear'] - 1)
resilience['headway_delta_pct'] = 100 * (resilience['headway_p50_rain'] / resilience['headway_p50_clear'] - 1)
resilience['speed_delta_pct'] = 100 * (resilience['speed_p50_rain'] / resilience['speed_p50_clear'] - 1)
resilience['gap_delta'] = resilience['service_gap_index_rain'] - resilience['service_gap_index_clear']
resilience['schedule_ratio_delta'] = resilience['observed_vs_scheduled_trip_ratio_rain'] - resilience['observed_vs_scheduled_trip_ratio_clear']

feature_cols = ['demand_delta_pct', 'headway_delta_pct', 'speed_delta_pct', 'gap_delta', 'schedule_ratio_delta']
scored = resilience.dropna(subset=feature_cols).copy()
scaler = StandardScaler()
scaled = scaler.fit_transform(scored[feature_cols])
scored['resilience_index'] = (
    -scaled[:, 0]
    -scaled[:, 1]
    +scaled[:, 2]
    -scaled[:, 3]
    +scaled[:, 4]
) / 5
scored['alternative_index'] = (
    -scored['demand_delta_pct'].rank(pct=True)
    -scored['headway_delta_pct'].rank(pct=True)
    +scored['speed_delta_pct'].rank(pct=True)
    -scored['gap_delta'].rank(pct=True)
    +scored['schedule_ratio_delta'].rank(pct=True)
) / 5

rank_corr = spearmanr(scored['resilience_index'], scored['alternative_index']).statistic
print('=== Route-level resilience panel ===')
print('routes with clear+rain coverage:', len(scored))
print('Spearman correlation between main and alternative index:', rank_corr)
print()
print(scored[['route_norm', 'operator', 'resilience_index', 'alternative_index']].sort_values('resilience_index', ascending=False).head(10).to_string(index=False))


In [ ]:
k = min(3, len(scored)) if len(scored) else 1
if len(scored):
    clusters = KMeans(n_clusters=max(k, 1), n_init=20, random_state=20260702).fit_predict(scaled)
    scored['cluster'] = clusters.astype(str)
else:
    scored['cluster'] = []

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
best = scored.sort_values('resilience_index', ascending=False).head(10)
worst = scored.sort_values('resilience_index', ascending=True).head(10)
sns.barplot(data=best, x='route_norm', y='resilience_index', hue='operator', ax=axes[0])
axes[0].set_title('Top resilient routes')
axes[0].tick_params(axis='x', rotation=45)
sns.barplot(data=worst, x='route_norm', y='resilience_index', hue='operator', ax=axes[1])
axes[1].set_title('Top vulnerable routes')
axes[1].tick_params(axis='x', rotation=45)
savefig('q4_resilience_rankings.png')
plt.show()

fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(data=scored, x='demand_delta_pct', y='headway_delta_pct', hue='cluster', style='operator', s=90, ax=ax)
ax.axhline(0, color='black', linewidth=1, linestyle='--')
ax.axvline(0, color='black', linewidth=1, linestyle='--')
ax.set_title('Q4 · Route clusters in resilience feature space')
ax.set_xlabel('Demand delta (%)')
ax.set_ylabel('Headway delta (%)')
savefig('q4_route_clusters.png')
plt.show()


## Leitura preliminar

- A rota só entra no ranking se tiver observações tanto em **tempo firme** quanto em **hora chuvosa**.
- O índice principal foi comparado com uma definição alternativa; a estabilidade é reportada pela correlação de Spearman.
- As rotas permanecem marcadas pelo `operator` e pelo `route_crosswalk_confidence`, para evitar superinterpretação de matches manuais.
